# Muon vs SGD: $\tau$ Scheduling (Joint Training vs Segment Alternation)

Consider a two-layer linear network $\hat y = W_2 W_1 x$, with target $W_2 W_1 \approx A$ ($d=64$, `seed=0`).

Throughout, we optimize the composition loss
$$L = \|W_2 W_1 - A\|^2 / \|A\|^2$$
until $\|W_2 W_1 - A\| / \|A\| \le \text{THRESHOLD}$.

**$\tau$ scheduling** (number of steps for which the other layer is frozen):
- $\tau = 0$: update $W_1$ and $W_2$ simultaneously at every step -> **joint GD** (green)
- $\tau \ge 1$: alternate segments: train only $W_1$ for $\tau$ steps (blue), then only $W_2$ for $\tau$ steps (red)

**Compared paths**:
- **SGD path**: $W_1$ uses SGD and $W_2$ uses SGD (solid line in the figures)
- **Muon path**: $W_1$ uses Muon and $W_2$ uses SGD (dashed line in the figures)

Both paths train $W_2$ with SGD; the only difference is the optimizer used for $W_1$.

Muon uses the same learning rate as SGD. It applies NS orthogonalization to the gradient, with update norm lr $\|g\|$; `NS_STEPS=5`.


In [ ]:
import math

import matplotlib.pyplot as plt
import torch

%matplotlib inline

torch.set_default_dtype(torch.float64)

D = 64
LR = 2e-2
SEED = 0
THRESHOLD = 0.05
MAX_STEPS = 41200
NS_STEPS = 5
TAU_VALUES = [0, 1, 10, 50, 200, 600]

PHASE_STYLE = {
    "joint": {"color": "#16a34a", "label": "joint (W1+W2)"},
    "w1": {"color": "#2563eb", "label": "W1 update"},
    "w2": {"color": "#dc2626", "label": "W2 update"},
}


In [ ]:
def zeropower_via_newtonschulz5(G, steps=3, eps=1e-7):
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X = X / (X.norm() + eps)
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(dtype=G.dtype)


def muon_direction(g, ns_steps=NS_STEPS, eps=1e-7):
    direction = zeropower_via_newtonschulz5(g, steps=ns_steps)
    return direction * (g.norm() / (direction.norm() + eps))


class Muon(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3, ns_steps=NS_STEPS):
        super().__init__(params, dict(lr=lr, ns_steps=ns_steps))

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            ns_steps = group["ns_steps"]
            for p in group["params"]:
                g = p.grad
                if g is None:
                    continue
                p.data.add_(muon_direction(g, ns_steps=ns_steps), alpha=-lr)


In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_target(d, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.randn(d, d, generator=gen) / math.sqrt(d)


def compose_loss(W2, W1, A, norm_A):
    M = W2 @ W1 - A
    return (M ** 2).sum() / (norm_A**2)


def rand_matrix(d, seed):
    state = torch.cuda.get_rng_state()
    torch.cuda.manual_seed(seed)
    W = torch.randn(d, d, device="cuda") / math.sqrt(d)
    torch.cuda.set_rng_state(state)
    return W


def step_joint(W1, W2, A, norm_A, opt1, opt2):
    opt1.zero_grad(set_to_none=True)
    opt2.zero_grad(set_to_none=True)
    compose_loss(W2, W1, A, norm_A).backward()
    opt1.step()
    opt2.step()


def step_w1_compose(W1, W2, A, norm_A, opt1):
    opt1.zero_grad(set_to_none=True)
    compose_loss(W2, W1, A, norm_A).backward()
    opt1.step()


def step_w2_compose(W1, W2, A, norm_A, opt2):
    opt2.zero_grad(set_to_none=True)
    compose_loss(W2, W1, A, norm_A).backward()
    opt2.step()


In [ ]:
def train_tau(
    A,
    w1_optimizer_class,
    tau=0,
    max_steps=MAX_STEPS,
    threshold=THRESHOLD,
    lr=LR,
    seed=SEED,
    ns_steps=NS_STEPS,
):
    set_seed(seed)
    W1 = rand_matrix(D, seed)
    W2 = rand_matrix(D, seed + 1)
    norm_A = A.norm().item()

    losses, phases = [], []
    if w1_optimizer_class is Muon:
        opt1 = w1_optimizer_class([W1], lr=lr, ns_steps=ns_steps)
    else:
        opt1 = w1_optimizer_class([W1], lr=lr)
    opt2 = torch.optim.SGD([W2], lr=lr)
    W1.requires_grad_(True)
    W2.requires_grad_(True)

    step = 0

    def record(phase):
        losses.append(compose_loss(W2, W1, A, norm_A).item())
        phases.append(phase)

    w1_opt = "muon" if w1_optimizer_class is Muon else "sgd"

    record("init")
    if losses[-1] <= threshold ** 2:
        return _pack(losses, phases, tau, w1_opt)

    if tau == 0:
        while step < max_steps and losses[-1] > threshold ** 2:
            step_joint(W1, W2, A, norm_A, opt1, opt2)
            step += 1
            record("joint")
        return _pack(losses, phases, tau, w1_opt)

    seg = 0
    while step < max_steps and losses[-1] > threshold ** 2:
        if seg % 2 == 0:
            for _ in range(tau):
                if step >= max_steps or losses[-1] <= threshold ** 2:
                    break
                step_w1_compose(W1, W2, A, norm_A, opt1)
                step += 1
                record("w1")
        else:
            for _ in range(tau):
                if step >= max_steps or losses[-1] <= threshold ** 2:
                    break
                step_w2_compose(W1, W2, A, norm_A, opt2)
                step += 1
                record("w2")
        seg += 1

    return _pack(losses, phases, tau, w1_opt)


def _pack(losses, phases, tau, w1_opt):
    return dict(
        losses=losses,
        phases=phases,
        tau=tau,
        w1_opt=w1_opt,
        steps=len(losses) - 1,
    )


def run_label(run):
    return "joint" if run["tau"] == 0 else f"tau={run['tau']}"


def converged(run):
    return run["losses"][-1] <= THRESHOLD ** 2


def delta_steps(sgd_run, muon_run):
    if not converged(sgd_run) or not converged(muon_run):
        return None
    return sgd_run["steps"] - muon_run["steps"]


def run_compare_sweep(A, tau_values, ns_steps=NS_STEPS):
    pairs = []
    print("\n" + "=" * 72)
    print("Compare SGD W1 vs Muon W1 (W2 always SGD)")
    print(f"  lr={LR}  ns_steps={ns_steps}")
    print("=" * 72)
    for tau in tau_values:
        sgd_run = train_tau(A, torch.optim.SGD, tau=tau)
        muon_run = train_tau(A, Muon, tau=tau, ns_steps=ns_steps)
        label = run_label(sgd_run)
        pairs.append((label, sgd_run, muon_run))
        d = delta_steps(sgd_run, muon_run)
        print(f"\n  [{label}]")
        print(f"    SGD W1 -> SGD W2:  steps={sgd_run['steps']}")
        print(f"    Muon W1 -> SGD W2: steps={muon_run['steps']}")
        print(f"    delta={d} (positive => Muon faster)")
    return pairs


In [ ]:
def plot_phased_loss(ax, run, line_style="-", linewidth=1.4, alpha=0.9):
    xs = list(range(len(run["losses"])))
    ys = run["losses"]
    phases = run["phases"]
    seen = set()
    for k in range(1, len(xs)):
        ph = phases[k]
        if ph not in PHASE_STYLE:
            continue
        style = PHASE_STYLE[ph]
        label = None
        if ph not in seen:
            label = style["label"]
            seen.add(ph)
        ax.plot(
            xs[k - 1:k + 1], ys[k - 1:k + 1],
            line_style, color=style["color"],
            linewidth=linewidth, alpha=alpha, label=label,
        )


def plot_phased_gap(ax, sgd_run, muon_run, xlim=None):
    n = min(len(sgd_run["losses"]), len(muon_run["losses"]))
    if xlim is not None:
        n = min(n, xlim[1] + 1)
    gaps = [sgd_run["losses"][k] - muon_run["losses"][k] for k in range(n)]
    phases = sgd_run["phases"]
    seen = set()
    for k in range(1, n):
        ph = phases[k]
        if ph not in PHASE_STYLE:
            continue
        style = PHASE_STYLE[ph]
        label = None
        if ph not in seen:
            label = style["label"]
            seen.add(ph)
        ax.plot(
            [k - 1, k], [gaps[k - 1], gaps[k]],
            color=style["color"], linewidth=1.5, label=label,
        )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("step")
    ax.set_ylabel(r"$L_{\rm SGD} - L_{\rm Muon}$" + "\n(negative => SGD lower)")
    ax.grid(True, alpha=0.3)


def plot_compare_panel(pairs, yscale="log", ns_steps=NS_STEPS, xlim=None, title_suffix=""):
    n = len(pairs)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 4.5 * rows), squeeze=False)
    scale_note = "linear y" if yscale == "linear" else "log y"
    fig.suptitle(
        f"Compose loss ({scale_note}){title_suffix}  |  lr={LR}  ns={ns_steps}\n"
        r"SGD solid, Muon dashed; green=joint, blue=W1, red=W2",
        fontsize=11,
    )
    loss_thresh = THRESHOLD ** 2
    for ax, (label, sgd_run, muon_run) in zip(axes.flat, pairs):
        plot_phased_loss(ax, sgd_run, line_style="-", linewidth=1.5)
        plot_phased_loss(ax, muon_run, line_style="--", linewidth=1.5, alpha=0.85)
        ax.axhline(loss_thresh, color="gray", linestyle=":", linewidth=0.8)
        ax.set_title(label)
        ax.set_xlabel("step")
        ax.set_ylabel(r"$L = \|W_2 W_1 - A\|^2 / \|A\|^2$")
        if yscale == "log":
            ax.set_yscale("log")
        if xlim is not None:
            ax.set_xlim(xlim)
        ax.grid(True, which="both", alpha=0.25)
    for ax in axes.flat[len(pairs):]:
        ax.axis("off")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


def plot_gap_panel(pairs, ns_steps=NS_STEPS, xlim=None, title_suffix=""):
    n = len(pairs)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 4 * rows), squeeze=False)
    zoom = f"  steps 0–{xlim[1]}" if xlim else ""
    fig.suptitle(
        f"Loss gap SGD − Muon{zoom}{title_suffix}  |  lr={LR}  ns={ns_steps}\n"
        r"$<0$: SGD ahead early   $>0$: Muon ahead late",
        fontsize=11,
    )
    for ax, (label, sgd_run, muon_run) in zip(axes.flat, pairs):
        plot_phased_gap(ax, sgd_run, muon_run, xlim=xlim)
        ax.set_title(label)
        if xlim is not None:
            ax.set_xlim(xlim)
    for ax in axes.flat[len(pairs):]:
        ax.axis("off")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


In [ ]:
set_seed(SEED)
A = make_target(D, seed=SEED).cuda()
pairs = run_compare_sweep(A, TAU_VALUES, ns_steps=NS_STEPS)


## Figure 1: Full Log Loss

The vertical axis is the composition loss on a log scale. Solid lines use SGD for W1, dashed lines use Muon for W1, and colors indicate which layer is currently being updated.


In [ ]:
plot_compare_panel(pairs, yscale="log", ns_steps=NS_STEPS)


## Figure 2: Linear Loss During the First 2000 Steps

A linear y-axis zooms in on the early stage, making it easier to see whether SGD is slightly lower than Muon during the blue W1-only segments.


In [ ]:
plot_compare_panel(pairs, yscale="linear", ns_steps=NS_STEPS, xlim=(0, 2000))


## Figure 3: Gap Curve (SGD - Muon Loss)

- **y < 0**: SGD loss is lower
- **y > 0**: Muon loss is lower
- **zero crossing**: the point where Muon overtakes SGD


In [ ]:
plot_gap_panel(pairs, ns_steps=NS_STEPS)
plot_gap_panel(pairs, ns_steps=NS_STEPS, xlim=(0, 2000))


## Summary

**Why Muon can gain an advantage:** early in training, during the blue segments where only $W_1$ is updated, Muon often reduces the loss more slowly per step than SGD. In that local W1-only update, it pays a speed cost. However, Muon's update behaves more like an orthogonal transformation, so the resulting $W_1$ provides a better representation for the downstream layer. The red segments, where only $W_2$ is updated, therefore become easier to optimize. After the red segment learns a better $W_2$, it in turn gives the upstream blue segment an easier $W_1$ subproblem. This cycle, "slower blue segment -> faster red segment -> easier next blue subproblem," compensates for Muon's weaker single-step W1 optimization and can still yield faster overall convergence.


## Why Muon's Blue Segments Can Become Faster Than SGD's in the Middle and Late Stages

During blue segments, Muon orthogonalizes the update, so on a single step it is often slower than SGD. Why, then, can the blue segments on the Muon path catch up with, or even become faster than, the blue segments on the SGD path in the middle and late stages?

The figure below ($\tau=200$) records the spectral flatness of $W_2$ on both paths at the **start of each blue segment** (larger means flatter). The Muon path often has higher spectral flatness: the spectrum of $W_2$ is flatter, which is equivalent to giving the currently updated $W_1$ an **easier subproblem**. Therefore, although Muon still pays a per-step speed cost from orthogonalization in the blue segments, it can gradually overtake SGD in the middle and late stages because the subproblem itself is better conditioned.

In the figure below, gap = $L_{\mathrm{SGD}} - L_{\mathrm{Muon}}$; $y < 0$ means the SGD loss is lower. Blue segments update only $W_1$.

- Gap decreases during a blue segment: $L_{\mathrm{SGD}}$ decreases faster than $L_{\mathrm{Muon}}$, so SGD extends its lead.
- Gap increases during a blue segment: $L_{\mathrm{Muon}}$ decreases relatively faster, so Muon closes the gap.

Orange dashed line: at the start of each blue segment, take the Muon path's $(W_1, W_2)$ and train only $W_1$ with SGD within that segment, plotting the resulting gap trajectory. This is meant to show that Muon's middle-to-late-stage comeback is mainly not because Muon is faster than SGD within the blue segment, but because the **subproblem on the Muon path is itself easier**. If SGD solves that same subproblem, it also descends faster.

In [ ]:
sgd_t200 = next(r for lbl, r, _ in pairs if lbl == "tau=200")
muon_t200 = next(r for _, _, r in pairs if r["tau"] == 200)
n = min(len(sgd_t200["losses"]), len(muon_t200["losses"]))
gaps = [sgd_t200["losses"][k] - muon_t200["losses"][k] for k in range(n)]
phases, sgd_loss = sgd_t200["phases"], sgd_t200["losses"]
TAU, PLOT_MAX = 200, 6000
norm_A = A.norm().item()


def w2_spectrum_flatness(W2):
    s = torch.linalg.svdvals(W2)
    p = s.pow(2)
    p = p / p.sum().clamp_min(1e-30)
    return torch.exp(-(p * torch.log(p.clamp_min(1e-30))).sum()).item() / D


def init_path(w1_cls):
    set_seed(SEED)
    W1, W2 = rand_matrix(D, SEED), rand_matrix(D, SEED + 1)
    kw = dict(ns_steps=NS_STEPS) if w1_cls is Muon else {}
    opt1 = w1_cls([W1], lr=LR, **kw)
    opt2 = torch.optim.SGD([W2], lr=LR)
    W1.requires_grad_(True)
    W2.requires_grad_(True)
    return W1, W2, opt1, opt2


def tau_replay(w1_cls, swap=False, gap_buf=None):
    W1, W2, opt1, opt2 = init_path(w1_cls)
    L = lambda W1_, W2_: compose_loss(W2_, W1_, A, norm_A).item()
    stop, step, seg, flat_tr = THRESHOLD ** 2, 0, 0, []
    while step < PLOT_MAX and L(W1, W2) > stop:
        if seg % 2 == 0:
            flat_tr.append((step, w2_spectrum_flatness(W2)))
            if gap_buf is not None and step < len(gap_buf):
                gap_buf[step] = sgd_loss[step] - L(W1, W2)
            W1s = opt_s = None
            if swap:
                W1s = W1.detach().clone()
                W1s.requires_grad_(True)
                opt_s = torch.optim.SGD([W1s], lr=LR)
                W2s = W2.detach().clone()
            for _ in range(TAU):
                if step >= PLOT_MAX or L(W1, W2) <= stop:
                    break
                if swap:
                    step_w1_compose(W1s, W2s, A, norm_A, opt_s)
                step_w1_compose(W1, W2, A, norm_A, opt1)
                step += 1
                if swap and gap_buf is not None and step < len(gap_buf):
                    gap_buf[step] = sgd_loss[step] - L(W1s, W2s)
        else:
            for _ in range(TAU):
                if step >= PLOT_MAX or L(W1, W2) <= stop:
                    break
                step_w2_compose(W1, W2, A, norm_A, opt2)
                step += 1
        seg += 1
    return flat_tr


def blue_segment_deltas(lo, hi):
    rows, k = [], lo
    while k <= hi and k < n:
        if phases[k] != "w1":
            k += 1
            continue
        s = k
        while k <= hi and k < n and phases[k] == "w1":
            k += 1
        if (e := k - 1) > s:
            rows.append(gaps[e] - gaps[s - 1])
    return rows


gap_swap = [float("nan")] * min(PLOT_MAX + 1, n)
sgd_tr = tau_replay(torch.optim.SGD)
muon_tr = tau_replay(Muon, swap=True, gap_buf=gap_swap)

fig, ax = plt.subplots(figsize=(10, 3.5))
for tr, sty, c in [(sgd_tr, "-", "#dc2626"), (muon_tr, "--", "#2563eb")]:
    xs, ys = zip(*tr)
    ax.plot(xs, ys, "o" + sty, color=c, linewidth=1.2, markersize=4,
            label="SGD path" if sty == "-" else "Muon path")
ax.set_xlim(0, PLOT_MAX)
ax.set_xlabel("step (W1 segment start)")
ax.set_ylabel(r"flatness($W_2$) at W1 segment start")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print("W1 segments (flatness at segment start):")
for (step, fs), (_, fm) in zip(sgd_tr, muon_tr):
    dg = gaps[step + TAU] - gaps[step] if step + TAU < n else None
    print(f"  {step}-{step + TAU}: flat SGD={fs:.4f}  Muon={fm:.4f}  dgap={dg:+.5f}  ({'Muon higher' if fm > fs else 'SGD higher'})")
print()

fig, ax = plt.subplots(figsize=(10, 3.5))
for k in range(1, min(PLOT_MAX, n)):
    w1 = phases[k] == "w1"
    ax.plot([k - 1, k], [gaps[k - 1], gaps[k]], color="#2563eb" if w1 else "#fecaca",
            linewidth=1.5 if w1 else 0.5, alpha=0.9 if w1 else 0.35)
swap_labeled = False
k = 1
while k < min(PLOT_MAX, len(gap_swap)):
    if phases[k] != "w1":
        k += 1
        continue
    xs, ys = [], []
    if not math.isnan(gap_swap[k - 1]):
        xs, ys = [k - 1], [gap_swap[k - 1]]
    while k < min(PLOT_MAX, len(gap_swap)) and phases[k] == "w1":
        if not math.isnan(gap_swap[k]):
            xs.append(k)
            ys.append(gap_swap[k])
        k += 1
    if len(xs) >= 2:
        ax.plot(xs, ys, color="#f97316", linestyle=(0, (6, 4)), linewidth=2.0,
                label="Muon state, SGD on W1" if not swap_labeled else None, zorder=3)
        swap_labeled = True
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlim(0, PLOT_MAX)
ax.set_xlabel("step")
ax.set_ylabel(r"gap = $L_{\rm SGD} - L_{\rm Muon}$  ($<0$: SGD lower)")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

mid = PLOT_MAX // 2
for label, lo, hi in [("early", 1, mid), ("later", mid, PLOT_MAX)]:
    rows = blue_segment_deltas(lo, hi)
    print(f"{label} W1 segments: {sum(d < 0 for d in rows)} gap down  {sum(d > 0 for d in rows)} gap up\n")
